In [ ]:
# parameters
input_image = None


In [3]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from radiomics import featureextractor
import SimpleITK as sitk
from tqdm import tqdm
import cv2

# ---- 配置路径 ----
PROJECT_ROOT = Path(r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main")
OUT_DIR = PROJECT_ROOT / "checkpoints" / "OCTnext"
TEST_VIS_DIR = OUT_DIR / "test_vis"
RADIOMICS_RESULTS_DIR = OUT_DIR / "radiomics_results"
RADIOMICS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- 定义类别映射 ----
class_mapping = {
    1: "SRF",      # 红色
    2: "PED",      # 蓝色
    3: "IRF",      # 绿色
    4: "SHRM",     # 黄色
    5: "EZ_defect" # 洋红色
}

# ---- 配置PyRadiomics提取器 ----
# 创建一个特征提取器实例
extractor = featureextractor.RadiomicsFeatureExtractor()

# 配置提取参数（可以根据需要调整）
extractor.settings.update({
    'binWidth': 25,
    'normalize': True,
    'normalizeScale': 100,
    'removeOutliers': True,
    'resampledPixelSpacing': None,  # 如果没有像素间距信息，设置为None
    'interpolator': sitk.sitkBSpline,
    'label': 255  # 标签值，我们会在处理时为每个类别创建二值掩膜
})

# 启用所有特征类
extractor.enableAllFeatures()

# ---- 处理函数 ----
def resize_mask_to_match_image(mask, reference_image):
    """
    将掩膜调整为与参考图像相同的尺寸
    """
    # 获取参考图像的尺寸
    ref_size = reference_image.GetSize()
    
    # 创建重采样器
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_image)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)  # 使用最近邻插值保持标签值
    resampler.SetOutputPixelType(sitk.sitkUInt8)
    
    # 执行重采样
    resized_mask = resampler.Execute(mask)
    
    return resized_mask

def extract_radiomics_features(image_path, mask_path, class_id, class_name):
    """
    为单个图像和特定类别的掩膜提取影像组学特征
    """
    try:
        # 读取图像和掩膜
        image = sitk.ReadImage(str(image_path))
        mask = sitk.ReadImage(str(mask_path))
        
        # 检查图像和掩膜尺寸是否匹配
        if image.GetSize() != mask.GetSize():
            print(f"调整掩膜尺寸以匹配图像: {image.GetSize()} -> {mask.GetSize()}")
            mask = resize_mask_to_match_image(mask, image)
        
        # 检查掩膜中是否存在当前类别
        mask_array = sitk.GetArrayFromImage(mask)
        if class_id not in np.unique(mask_array):
            return None  # 如果掩膜中没有这个类别，跳过
        
        # 提取特征，指定标签为当前类别ID
        features = extractor.execute(image, mask, label=class_id)
        
        # 过滤掉诊断信息，只保留特征值
        feature_dict = {}
        for key, value in features.items():
            if not key.startswith('diagnostics'):
                feature_dict[key] = value
        
        return feature_dict
        
    except Exception as e:
        print(f"处理 {image_path} 的 {class_name} 区域时出错: {str(e)}")
        return None

# ---- 主处理流程 ----
def main():
    # 获取所有测试图像
    test_images_dir = Path(r"F:\AMD-SD\AMD-SD\test")
    image_extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    image_paths = []
    
    for ext in image_extensions:
        image_paths.extend(list(test_images_dir.rglob(f"*{ext}")))
        image_paths.extend(list(test_images_dir.rglob(f"*{ext.upper()}")))
    
    print(f"找到 {len(image_paths)} 张测试图像")
    
    # 准备结果数据框架
    all_features = []
    
    # 处理每张图像
    for image_path in tqdm(image_paths, desc="处理图像"):
        image_name = image_path.stem
        
        # 查找对应的掩膜文件
        mask_path = TEST_VIS_DIR / "idmask" / f"{image_name}_id.png"
        
        if not mask_path.exists():
            print(f"警告: 未找到 {image_name} 的掩膜文件")
            continue
        
        # 为每个类别提取特征
        for class_id, class_name in class_mapping.items():
            features = extract_radiomics_features(
                image_path, mask_path, class_id, class_name
            )
            
            if features is not None:
                # 添加元数据
                features["Image"] = image_name
                features["Class"] = class_name
                features["Class_ID"] = class_id
                
                all_features.append(features)
    
    # 转换为DataFrame并保存
    if all_features:
        df = pd.DataFrame(all_features)
        
        # 重新排列列，将元数据放在前面
        cols = ["Image", "Class", "Class_ID"] + [col for col in df.columns if col not in ["Image", "Class", "Class_ID"]]
        df = df[cols]
        
        # 保存结果
        output_path = RADIOMICS_RESULTS_DIR / "radiomics_features.csv"
        df.to_csv(output_path, index=False)
        print(f"影像组学特征已保存到: {output_path}")
        
        # 也保存每个类别的单独文件
        for class_name in class_mapping.values():
            class_df = df[df["Class"] == class_name]
            if not class_df.empty:
                class_output_path = RADIOMICS_RESULTS_DIR / f"radiomics_features_{class_name}.csv"
                class_df.to_csv(class_output_path, index=False)
        
        print(f"共提取了 {len(df)} 个样本的特征")
    else:
        print("未成功提取任何特征")

# ---- 可视化函数（可选）----
def visualize_roi(image_path, mask_path, class_id, output_dir):
    """
    可视化特定类别的ROI区域
    """
    try:
        # 读取图像和掩膜
        image = cv2.imread(str(image_path))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        # 如果尺寸不匹配，调整掩膜尺寸
        if image.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        
        # 创建特定类别的二值掩膜
        binary_mask = (mask == class_id).astype(np.uint8) * 255
        
        # 在图像上绘制ROI轮廓
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        roi_image = image.copy()
        cv2.drawContours(roi_image, contours, -1, (0, 255, 0), 2)
        
        # 保存可视化结果
        image_name = Path(image_path).stem
        class_name = class_mapping[class_id]
        output_path = output_dir / f"{image_name}_{class_name}_roi.png"
        cv2.imwrite(str(output_path), roi_image)
    except Exception as e:
        print(f"可视化 {image_path} 的 {class_mapping[class_id]} ROI时出错: {str(e)}")

# ---- 运行主函数 ----
if __name__ == "__main__":
    main()
    
    # 可选：创建ROI可视化
    roi_vis_dir = RADIOMICS_RESULTS_DIR / "roi_visualization"
    roi_vis_dir.mkdir(exist_ok=True)
    
    # 为每个图像和类别创建ROI可视化
    test_images_dir = Path(r"F:\AMD-SD\AMD-SD\test")
    image_extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    image_paths = []
    
    for ext in image_extensions:
        image_paths.extend(list(test_images_dir.rglob(f"*{ext}")))
    
    for image_path in tqdm(image_paths, desc="创建ROI可视化"):
        image_name = image_path.stem
        mask_path = TEST_VIS_DIR / "idmask" / f"{image_name}_id.png"
        
        if mask_path.exists():
            for class_id in class_mapping.keys():
                visualize_roi(image_path, mask_path, class_id, roi_vis_dir)
    
    print(f"ROI可视化已保存到: {roi_vis_dir}")

找到 388 张测试图像


处理图像:   0%|                                                                                | 0/388 [00:00<?, ?it/s]Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
处理图像:   0%|▏                                                                       | 1/388 [00:00<00:58,  6.65it/s]

调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)


Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
处理图像:   1%|▎                                                                       | 2/388 [00:00<00:59,  6.48it/s]Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)


...

调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)


Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
处理图像: 100%|██████████████████████████████████████████████████████████████████████| 388/388 [01:28<00:00,  4.39it/s]


调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
调整掩膜尺寸以匹配图像: (570, 380) -> (512, 512)
影像组学特征已保存到: F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext\radiomics_results\radiomics_features.csv
共提取了 1530 个样本的特征


创建ROI可视化: 100%|█████████████████████████████████████████████████████████████████| 194/194 [00:13<00:00, 14.01it/s]

ROI可视化已保存到: F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext\radiomics_results\roi_visualization


In [3]:
# -*- coding: utf-8 -*-
# ================== 仅对“保留下来的 overlay 样本”提取影像组学特征（含ROI可视化 & 更强路径解析 & 响应变量） ==================
import os
import re
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import SimpleITK as sitk
from radiomics import featureextractor
from typing import Optional, Tuple

# -------------------- 路径配置 --------------------
PROJECT_ROOT = Path(r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main")
OUT_DIR = PROJECT_ROOT / "checkpoints" / "OCTnext"
TEST_VIS_DIR = OUT_DIR / "test_vis"   # 你保留overlay的根目录（现在其下一层可能是 “响应/未响应” 等）
RADIOMICS_RESULTS_DIR = OUT_DIR / "radiomics_results"
RADIOMICS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 原始图像搜索目录（先在 F:\OCT 里找，找不到再去 AMD-SD 测试集里兜底）
PRIMARY_IMAGE_ROOT = Path(r"F:\OCT")
SECONDARY_IMAGE_ROOT = Path(r"F:\AMD-SD\AMD-SD\test")

# -------------------- 类别映射（与训练一致） --------------------
class_mapping = {
    1: "SRF",      # 红色
    2: "PED",      # 蓝色
    3: "IRF",      # 绿色
    4: "SHRM",     # 黄色
    5: "EZ_defect" # 洋红色
}

# 画轮廓用的颜色（BGR）
class_bgr = {
    1: (0, 0, 255),    # SRF - 红
    2: (255, 0, 0),    # PED - 蓝
    3: (0, 255, 0),    # IRF - 绿
    4: (0, 255, 255),  # SHRM - 黄
    5: (255, 0, 255),  # EZ_defect - 洋红
}

# -------------------- 药物字典（用于路径解析） --------------------
drug_en2cn = {
    "Aflibercept": "阿柏西普",
    "Faricimab":   "法瑞西单抗",
    "Conbercept":  "康柏西普",
    "Ranibizumab": "雷珠单抗"
}
# 扩展匹配（英文与中文的集合）
KNOWN_DRUG_TOKENS = set([k.lower() for k in drug_en2cn.keys()] + [v.lower() for v in drug_en2cn.values()])

# -------------------- 响应变量解析（多别名支持） --------------------
# 规范化后匹配：去掉空格/下划线/连字符并小写
def _norm_token(s: str) -> str:
    return re.sub(r"[\s_\-]+", "", str(s)).lower()

RESP_ALIASES = {
    "响应": "响应", "應答": "响应", "应答": "响应",
    "responder": "响应", "responders": "响应", "r": "响应",
    "未响应": "未响应", "不响应": "未响应", "無響應": "未响应", "无响应": "未响应",
    "nonresponder": "未响应", "nonresponders": "未响应", "nr": "未响应", "nonrespondergroup": "未响应"
}
RESP_ALIASES_NORM = { _norm_token(k): v for k, v in RESP_ALIASES.items() }

def find_response_from_parts(parts) -> Tuple[str, str]:
    """
    在路径片段中查找响应变量。返回 (raw_match, response)
    response ∈ {'响应','未响应','未知'}
    """
    for p in parts:
        norm = _norm_token(p)
        if norm in RESP_ALIASES_NORM:
            return str(p), RESP_ALIASES_NORM[norm]
        # 片段包含“响应/未响应”汉字的宽松匹配
        if "未响应" in str(p):
            return str(p), "未响应"
        if "响应" in str(p) and "未响应" not in str(p):
            return str(p), "响应"
    return "", "未知"

def response_to01(resp: str) -> int:
    if resp == "响应":
        return 1
    if resp == "未响应":
        return 0
    return -1  # 未知

# -------------------- PyRadiomics 提取器配置 --------------------
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings.update({
    'binWidth': 25,
    'normalize': True,
    'normalizeScale': 100,
    'removeOutliers': True,
    'resampledPixelSpacing': None,   # 2D PNG 通常没有物理间距，用 None
    'interpolator': sitk.sitkBSpline # 图像插值
    # 'label': 不必设，execute 时用 label=class_id
})
extractor.enableAllFeatures()

# -------------------- 工具函数 --------------------
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".PNG", ".JPG", ".JPEG", ".TIF", ".TIFF", ".BMP"}

def safe_imread(path: Path, flags=cv2.IMREAD_COLOR) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, flags)
    return img

def safe_imwrite(path: Path, img: np.ndarray) -> None:
    path = Path(path)
    img = np.ascontiguousarray(img)
    ext = path.suffix or ".png"
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        raise RuntimeError(f"cv2.imencode 失败：{path}")
    buf.tofile(str(path))

CN_NUM = {
    "零":0, "〇":0, "○":0, "O":0, "o":0,
    "一":1, "二":2, "两":2, "三":3, "四":4, "五":5, "六":6, "七":7, "八":8, "九":9, "十":10
}
def parse_chinese_numeral(s: str) -> int:
    """解析常见中文数字（十、十一、二十、二十三等），或阿拉伯数字"""
    s = s.strip()
    if s.isdigit():
        return int(s)
    if s == "十":
        return 10
    m = re.match(r"^十([一二两三四五六七八九])$", s)
    if m:
        return 10 + CN_NUM[m.group(1)]
    m = re.match(r"^([一二两三四五六七八九])十([一二两三四五六七八九])?$", s)
    if m:
        high = CN_NUM[m.group(1)] * 10
        low  = CN_NUM[m.group(2)] if m.group(2) else 0
        return high + low
    return CN_NUM.get(s, 0)

# 支持两种次序（含可选的空格/下划线/连字符分隔）：
# 1) 第N针 + 术前/术后
pat_after = re.compile(r"(?:第)?\s*([一二两三四五六七八九十\d]+)\s*针[\s_\-]*?(术前|术后)")
# 2) 术前/术后 + 第N针
pat_before = re.compile(r"(术前|术后)[\s_\-]*?(?:第)?\s*([一二两三四五六七八九十\d]+)\s*针")

def parse_injection_stage_from_parts(parts):
    """
    查找“第N针术前/术后”或“术前/术后第N针”，返回：
    raw, n, when, prior_injections, group_text, prepost('术前'/'术后'/'未知')
    """
    for p in parts:
        s = str(p)
        m1 = pat_after.search(s)
        if m1:
            n = parse_chinese_numeral(m1.group(1))
            when = m1.group(2)
            prior = max(0, n-1) if when == "术前" else max(0, n)
            group_text = "未用药" if prior == 0 else f"{prior}次后"
            return s, n, when, prior, group_text, when
        m2 = pat_before.search(s)
        if m2:
            when = m2.group(1)
            n = parse_chinese_numeral(m2.group(2))
            prior = max(0, n-1) if when == "术前" else max(0, n)
            group_text = "未用药" if prior == 0 else f"{prior}次后"
            return s, n, when, prior, group_text, when
    return "", None, "", 0, "未知", "未知"

def index_images(*roots: Path) -> dict:
    """建立 {stem: full_path} 索引，先来的优先"""
    idx = {}
    for root in roots:
        if not root or not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.suffix in IMG_EXTS:
                st = p.stem
                if st not in idx:
                    idx[st] = p
    return idx

def safe_read_sitk(path: Path, as_mask=False) -> sitk.Image:
    """优先 SimpleITK，失败回退 OpenCV+SITK；as_mask=True 时输出 uint8。"""
    try:
        img = sitk.ReadImage(str(path))
        if as_mask:
            img = sitk.Cast(img, sitk.sitkUInt8)
        return img
    except Exception:
        arr = safe_imread(path, cv2.IMREAD_GRAYSCALE)
        if arr is None:
            raise RuntimeError(f"无法读取图像: {path}")
        if as_mask:
            return sitk.GetImageFromArray(arr.astype(np.uint8))
        else:
            return sitk.GetImageFromArray(arr)

def ensure_scalar_image(image: sitk.Image) -> sitk.Image:
    """若为多通道图像，取第1通道"""
    if image.GetNumberOfComponentsPerPixel() > 1:
        image = sitk.VectorIndexSelectionCast(image, 1)
    return image

def resize_image_to_match_mask(image: sitk.Image, mask_ref: sitk.Image) -> sitk.Image:
    """将原始图像重采样到掩膜尺寸/空间"""
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(mask_ref)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetOutputPixelType(sitk.sitkFloat32)
    return resampler.Execute(image)

def extract_radiomics_features(image_path: Path, mask_path: Path, class_id: int) -> Optional[dict]:
    """
    对单个样本就某个 class_id 提取特征；返回 dict 或 None
    """
    try:
        image = safe_read_sitk(image_path, as_mask=False)
        image = ensure_scalar_image(image)
        mask  = safe_read_sitk(mask_path, as_mask=True)

        if image.GetSize() != mask.GetSize():
            image = resize_image_to_match_mask(image, mask)

        m_arr = sitk.GetArrayFromImage(mask)
        if (m_arr == class_id).sum() == 0:
            return None

        feats = extractor.execute(image, mask, label=class_id)
        return {k: v for k, v in feats.items() if not k.startswith("diagnostics")}
    except Exception as e:
        print(f"[WARN] 提取失败: {image_path.name} (class={class_id}) -> {e}")
        return None

# -------------------- 目录解析：药物与响应变量 --------------------
def find_drug_from_parts(parts) -> str:
    """
    在路径片段（通常为 overlay 之前的所有层级）中查找药物名称（中/英）。
    规则：任一片段包含已知药物 token 即视为命中；优先返回中文（若以英文命中则映射到中文）。
    """
    # 先尝试精确 token 命中
    for p in parts:
        pl = _norm_token(p)
        if pl in KNOWN_DRUG_TOKENS:
            # pl 可能是中文或英文
            for en, cn in drug_en2cn.items():
                if pl == en.lower() or pl == cn.lower():
                    return cn
    # 再尝试宽松包含（去除修饰后包含）
    for p in parts:
        raw = str(p).lower()
        for en, cn in drug_en2cn.items():
            if en.lower() in raw or cn.lower() in raw:
                return cn
    return "未知"

# -------------------- ROI 可视化 --------------------
def visualize_roi(image_path: Path, mask_path: Path, class_id: int, out_dir: Path):
    """
    在原图尺寸上绘制 class_id 的 ROI 轮廓；中文路径安全保存
    """
    try:
        img = safe_imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"[ROI] 读取原图失败：{image_path}")
            return
        mask = safe_imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"[ROI] 读取掩膜失败：{mask_path}")
            return

        # 对齐尺寸
        if img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

        binary = (mask == class_id).astype(np.uint8) * 255
        if binary.max() == 0:
            return  # 无该类

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        vis = img.copy()
        cv2.drawContours(vis, contours, -1, class_bgr.get(class_id, (0,255,0)), 2)

        out_dir.mkdir(parents=True, exist_ok=True)
        cname = class_mapping[class_id]
        stem = image_path.stem
        out_path = out_dir / f"{stem}_{cname}_roi.png"
        safe_imwrite(out_path, vis)
    except Exception as e:
        print(f"[ROI] 可视化失败: {image_path} ({class_id}) -> {e}")

# -------------------- 主流程 --------------------
def main():
    # 1) 原始图像索引
    stem2img = index_images(PRIMARY_IMAGE_ROOT, SECONDARY_IMAGE_ROOT)
    print(f"[INFO] 原始图像索引完成：{len(stem2img)} 个唯一文件名（stem）")

    # 2) 只收集“保留的 overlay”文件（兼容新目录，无需改）
    overlay_files = []
    for p in TEST_VIS_DIR.rglob("*"):
        if p.is_file() and p.suffix in IMG_EXTS and p.parent.name.lower() == "overlay":
            overlay_files.append(p)
    print(f"[INFO] 找到保留的 overlay 图像：{len(overlay_files)} 张")

    if not overlay_files:
        print("[ERROR] 未找到 overlay 图像，请检查目录。")
        return

    # 3) 遍历 overlay，配套 idmask 与 原始图片，解析药物/用药阶段/响应变量
    rows = []
    processed_pairs = set()  # 用于 ROI：去重 (ImagePath, MaskPath)
    for ov_path in tqdm(overlay_files, desc="提取特征（按保留overlay）"):
        rel = ov_path.relative_to(TEST_VIS_DIR)
        parts = rel.parts  # 例：['未响应', 'Aflibercept', '病例X', 'overlay', 'xxx_ov.png'] 或 ['Drug', 'overlay', 'xxx_ov.png'] 等

        # 文件名去掉 _ov
        stem = ov_path.stem
        if stem.endswith("_ov"):
            stem = stem[:-3]

        # idmask 同级目录
        idmask_dir = ov_path.parent.parent / "idmask"
        idmask_path = idmask_dir / f"{stem}_id.png"
        if not idmask_path.exists():
            cand = list(idmask_dir.glob(f"{stem}_id.*"))
            if cand:
                idmask_path = cand[0]
            else:
                print(f"[SKIP] 缺少掩膜: {idmask_path}")
                continue

        # 原始图像通过 stem 匹配
        image_path = stem2img.get(stem)
        if image_path is None:
            print(f"[SKIP] 找不到原始图像（stem={stem}），跳过。")
            continue

        # 在 'overlay' 之前的片段中解析“针次/术前术后”
        pre_overlay_parts = parts[:-2] if len(parts) >= 2 else parts
        stage_raw, n, when, prior_inj, group_text, prepost = parse_injection_stage_from_parts(pre_overlay_parts)

        # 解析响应变量（路径任意层级都可识别）
        resp_raw, resp = find_response_from_parts(parts)
        resp01 = response_to01(resp)

        # 解析药物（尽量在 overlay 之前的层级中找）
        drug = find_drug_from_parts(pre_overlay_parts)

        # 解析眼别 & 患者线索
        eye = ""
        for p in parts:
            if "左眼" in p: eye = "左眼"
            if "右眼" in p: eye = "右眼"

        # 生成 SubjectHint：尽量去除最可能的“药物片段/响应片段/针次片段”
        exclude_tokens = {drug, stage_raw, resp_raw}
        core_parts = []
        for p in pre_overlay_parts:
            if p not in exclude_tokens and p not in {"overlay", "idmask"}:
                core_parts.append(p)
        subject_hint = "/".join(core_parts) if core_parts else ""

        # 每个类别提取特征
        any_feat = False
        for class_id, class_name in class_mapping.items():
            feats = extract_radiomics_features(image_path, idmask_path, class_id)
            if feats is None:
                continue

            row = {
                "ImageStem": stem,
                "ImagePath": str(image_path),
                "MaskPath": str(idmask_path),
                "OverlayPath": str(ov_path),

                # ===== 新增：响应变量 =====
                "Response": resp,            # '响应' / '未响应' / '未知'
                "Response01": int(resp01),   # 1 / 0 / -1

                # ===== 信息列（药物不是响应变量，只做协变量/分层信息）=====
                "Drug": drug,                # '阿柏西普' 等或 '未知'

                # ===== 针次/分组列 =====
                "InjectionStageRaw": stage_raw if stage_raw else "未知",
                "PrePost": prepost if prepost else "未知",     # 术前/术后/未知
                "PriorInjections": int(prior_inj),
                "StageGroup": group_text,                      # 未用药 / 1次后 / 2次后 ...

                # ===== 其他元数据 =====
                "Eye": eye if eye else "未知",
                "SubjectHint": subject_hint if subject_hint else "未知",
                "Class": class_name,
                "Class_ID": class_id
            }
            for k, v in feats.items():
                row[k] = v
            rows.append(row)
            any_feat = True

        if any_feat:
            processed_pairs.add((str(image_path), str(idmask_path)))

    if not rows:
        print("[WARN] 未提取到任何有效特征。")
        return

    # 4) 保存特征表
    df = pd.DataFrame(rows)

    # 元数据列顺序（新增 Response/Response01 放前面）
    meta_cols = [
        "ImageStem",
        "Response","Response01",
        "Drug",
        "InjectionStageRaw","PrePost","PriorInjections","StageGroup",
        "Eye","SubjectHint",
        "Class","Class_ID",
        "ImagePath","MaskPath","OverlayPath"
    ]
    other_cols = [c for c in df.columns if c not in meta_cols]
    df = df[meta_cols + other_cols]

    out_all = RADIOMICS_RESULTS_DIR / "radiomics_features_kept_overlays.csv"
    df.to_csv(out_all, index=False, encoding="utf-8-sig")
    print(f"[OK] 已保存总表: {out_all}")

    # 按类别分别输出 CSV
    for cname in class_mapping.values():
        sub = df[df["Class"] == cname]
        if not sub.empty:
            out_c = RADIOMICS_RESULTS_DIR / f"radiomics_features_kept_overlays_{cname}.csv"
            sub.to_csv(out_c, index=False, encoding="utf-8-sig")

    # 简单统计：响应分布
    try:
        cnt = df["Response"].value_counts(dropna=False).to_dict()
        print(f"[INFO] 响应分布：{cnt}")
    except Exception:
        pass

    print(f"[DONE] 样本数（按行，类别展开后）: {len(df)}")
    print("      关键列：Response / Response01 / Drug / PrePost / PriorInjections / StageGroup 已写入。")

    # 5) 生成 ROI 可视化（仅对本次参与特征提取的样本；按类分别输出）
    roi_root = RADIOMICS_RESULTS_DIR / "roi_visualization"
    for image_path, mask_path in tqdm(sorted(processed_pairs), desc="生成ROI可视化"):
        image_path = Path(image_path)
        mask_path  = Path(mask_path)
        for cid in class_mapping.keys():
            out_dir = roi_root / class_mapping[cid]
            visualize_roi(image_path, mask_path, cid, out_dir)
    print(f"[OK] ROI 可视化已输出到: {roi_root}")

if __name__ == "__main__":
    main()


[INFO] 原始图像索引完成：4725 个唯一文件名（stem）
[INFO] 找到保留的 overlay 图像：456 张


提取特征（按保留overlay）:   0%|                                                               | 0/456 [00:00<?, ?it/s]Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
提取特征（按保留overlay）:   0%|                                                       | 1/456 [00:00<03:26,  2.21it/s]Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input

[WARN] 提取失败: 40DBF3E0.tif (class=1) -> mask only contains 1 segmented voxel! Cannot extract features for a single voxel.


Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
提取特征（按保留overlay）:   2%|▊                                                      | 7/456 [00:01<01:15,  5.96it/s]Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to

[WARN] 提取失败: 720C7D80.tif (class=3) -> mask only contains 1 segmented voxel! Cannot extract features for a single voxel.
[WARN] 提取失败: 72115F80.tif (class=3) -> mask only contains 1 segmented voxel! Cannot extract features for a single voxel.


...

[WARN] 提取失败: C56CEF10.tif (class=3) -> mask only contains 1 segmented voxel! Cannot extract features for a single voxel.


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
提取特征（按保留overlay）:  93%|█████████████████████████████████████████████████    | 422/456 [00:53<00:05,  6.18it/s]Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
Shape features are only available 3D input (for 2D input, use shape2D). 

[OK] 已保存总表: F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext\radiomics_results\radiomics_features_kept_overlays.csv
[INFO] 响应分布：{'未响应': 926, '响应': 182}
[DONE] 样本数（按行，类别展开后）: 1108
      关键列：Response / Response01 / Drug / PrePost / PriorInjections / StageGroup 已写入。


生成ROI可视化: 100%|█████████████████████████████████████████████████████████████████| 456/456 [00:25<00:00, 17.68it/s]


[OK] ROI 可视化已输出到: F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext\radiomics_results\roi_visualization


In [6]:
# -*- coding: utf-8 -*-
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

from skimage import exposure, io
from skimage.filters.rank import entropy
from skimage.morphology import disk
from skimage.transform import resize
from PIL import Image  # 兜底读取（例如 TIFF/LZW）

# ========= 0) 基础配置 =========
csv_path = r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext\radiomics_results\filtered_by_roi\radiomics_features_kept_overlays_IRF.filtered.csv"
out_dir  = os.path.join(os.path.dirname(csv_path), "entropy_overlays_IRF_IRFonly")
os.makedirs(out_dir, exist_ok=True)

# ========= 1) 读取 radiomics 表，定位列 =========
df = pd.read_csv(csv_path)

def pick(cols, cands):
    cl = [c for c in cols]
    for c in cands:
        hit = [x for x in cl if x.lower() == c.lower()]
        if hit: return hit[0]
    return None

col_img  = pick(df.columns, ["ImagePath","image_path","img","img_path","orig_img","orig_path"])
col_mask = pick(df.columns, ["MaskPath","mask_path","Mask","mask","roi_mask","irf_mask"])
col_y    = pick(df.columns, ["GroupResp","Response01","response","y","label","outcome"])
assert col_img and col_mask and col_y, "CSV 里需要包含：原图、掩膜、响应(GroupResp/response等)三列"

def to01(series):
    if pd.api.types.is_numeric_dtype(series):
        return (series >= 0.5).astype(int)
    t = series.astype(str).str.lower().str.strip()
    pos = set(["1","yes","y","true","t","responder","respond","positive","pos","阳性","响应","应答"])
    return t.isin(pos).astype(int)

df["y01"] = to01(df[col_y])

# ========= 2) 读取图像（鲁棒） & 8-bit 灰度转换 =========
def imread_robust(p):
    """优先 skimage；失败(如 LZW 缺 imagecodecs)则用 Pillow."""
    try:
        return io.imread(p)
    except Exception:
        with Image.open(p) as im:
            return np.array(im)

def to_gray_uint8(arr):
    """转灰度+拉伸到 uint8（rank.entropy 需要 8-bit）"""
    im = arr
    if im.ndim == 3:
        if im.shape[-1] >= 3:
            im = np.dot(im[..., :3], [0.299, 0.587, 0.114])
        else:
            im = im[..., 0]
    im = exposure.rescale_intensity(im, in_range='image', out_range=(0, 255)).astype(np.uint8)
    return im

# ========= 3) 从掩膜中得到 IRF 区域（只返回 IRF 布尔掩膜） =========
def get_irf_mask(mask_arr):
    """
    - 若为单通道整数标签：优先 label==3；否则视作二值掩膜 >0。
    - 若为 RGB 伪彩标签：IRF 约定为纯绿色(0,255,0)，做精确匹配。
    """
    m = mask_arr
    if m.ndim == 2:
        vals = np.unique(m)
        if 3 in vals:             # 多类整型标签
            return (m == 3)
        else:                      # 二值或其它：>0 视作 IRF
            return (m > 0)
    else:
        # RGB：严格匹配绿色 (0,255,0)
        return (m[..., 0] == 0) & (m[..., 1] == 255) & (m[..., 2] == 0)

# ========= 4) 选择各取一例（保证 IRF 非空） =========
def choose_one(df_sub):
    base_dir = os.path.dirname(csv_path)
    for _, row in df_sub.iterrows():
        img_p  = row[col_img]
        mask_p = row[col_mask]
        if not os.path.isabs(img_p):  img_p  = os.path.normpath(os.path.join(base_dir, img_p))
        if not os.path.isabs(mask_p): mask_p = os.path.normpath(os.path.join(base_dir, mask_p))
        if (not os.path.exists(img_p)) or (not os.path.exists(mask_p)):
            continue
        msk_raw = imread_robust(mask_p)
        irf = get_irf_mask(msk_raw)
        if np.any(irf):
            row = row.copy()
            row[col_img], row[col_mask] = img_p, mask_p
            return row
    return None

row_r  = choose_one(df[df["y01"] == 1])
row_nr = choose_one(df[df["y01"] == 0])
assert row_r is not None and row_nr is not None, "未找到 IRF 区域非空的响应/未响应样本，请检查掩膜或更换样本。"
pairs  = [("Responder", row_r), ("NonResponder", row_nr)]

# ========= 5) 仅 IRF 内部的“局部像素熵”叠加 =========
def overlay_entropy_irf_only(img_u8, irf_mask, radius_px=9,
                             vmin=0.0, vmax=np.log2(256),
                             title="", out_png=None, out_pdf=None,
                             cmap="inferno", alpha=0.85):
    """
    只在 IRF 区域内可视化局部熵；无任何轮廓或其它区域着色。
    """
    ent = entropy(img_u8, disk(radius_px))  # Shannon 熵，单位 bit（需要 8-bit 输入）
    ent_vis = ent.astype(float)
    ent_vis[~irf_mask] = np.nan

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.imshow(img_u8, cmap="gray")
    im = ax.imshow(ent_vis, cmap=cmap, alpha=alpha, vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Local entropy (bits)")
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    if out_png: fig.savefig(out_png, dpi=300)
    if out_pdf: fig.savefig(out_pdf)
    plt.close(fig)
    return ent

# ========= 6) 执行并保存 =========
global_min, global_max = 0.0, np.log2(256)  # 统一色标 [0, 8] bits
records = []
radius = 9  # 可调窗口半径（像素）

for tag, row in pairs:
    img_p, mask_p = row[col_img], row[col_mask]
    assert os.path.exists(img_p) and os.path.exists(mask_p), f"找不到：{img_p} 或 {mask_p}"

    img = imread_robust(img_p)
    msk = imread_robust(mask_p)

    img_u8 = to_gray_uint8(img)
    irf_mask = get_irf_mask(msk)

    # 尺寸不一致 -> NN 缩放掩膜以对齐原图
    if irf_mask.shape != img_u8.shape:
        irf_mask = resize(irf_mask.astype(float), img_u8.shape, order=0,
                          preserve_range=True, anti_aliasing=False) > 0.5

    base  = os.path.splitext(os.path.basename(img_p))[0]
    title = f"{tag}: IRF local entropy (r={radius}px)"
    outpng = os.path.join(out_dir, f"{base}_{tag}_IRF_entropy.png")
    outpdf = os.path.join(out_dir, f"{base}_{tag}_IRF_entropy.pdf")

    ent = overlay_entropy_irf_only(
        img_u8, irf_mask, radius_px=radius,
        vmin=global_min, vmax=global_max,
        title=title, out_png=outpng, out_pdf=outpdf
    )

    vals = ent[irf_mask]
    records.append({
        "case": base, "group": tag,
        "entropy_mean": float(np.nanmean(vals)),
        "entropy_median": float(np.nanmedian(vals)),
        "entropy_p95": float(np.nanpercentile(vals, 95)),
        "IRF_area_px": int(irf_mask.sum())
    })

pd.DataFrame.from_records(records).to_csv(
    os.path.join(out_dir, "IRF_entropy_summary_two_cases_IRFonly.csv"), index=False
)

print("[OK] 已输出：", out_dir)


[OK] 已输出： F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext\radiomics_results\filtered_by_roi\entropy_overlays_IRF_IRFonly
